In [0]:
#Load the JSON file into Bronze
#Read the JSON and save the raw copy (cell 1)
from pyspark.sql.functions import current_timestamp, lit, col

CATALOG = "capstone_supply_chain"

df_json_raw = (spark.read
    .option("multiLine", "true")   # the JSON spans many lines, not one record per line
    .json(f"/Volumes/{CATALOG}/bronze/landing_json/shipping_logistics.json")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))   # replaces input_file_name()
    .withColumn("_file_format", lit("json"))
)

# Look at the structure Spark discovered
df_json_raw.printSchema()

# Save the raw, nested version
df_json_raw.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.raw_shipping_raw")
print("✅ raw_shipping_raw rows:", df_json_raw.count())

In [0]:
#Flatten the shipments (cell 2)
from pyspark.sql.functions import explode, col

df_shipments = (df_json_raw
    .select(
        col("api_version"),
        col("extracted_at"),
        explode(col("shipments")).alias("shipment"),   # one row per shipment
        col("_ingested_at"),
        col("_source_file"),
        col("_file_format"),
    )
    .select(
        "api_version", "extracted_at",
        "shipment.*",                  # keep ALL shipment fields
        "_ingested_at", "_source_file", "_file_format",
    )
)

df_shipments.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.raw_shipping_logistics")
print("✅ raw_shipping_logistics shipments:", df_shipments.count())
display(df_shipments)